# GPT Action 库（中间件）：AWS Lambda

## 简介

这个 GPT Action 概述了如何构建 **AWS Lambda** 函数。本文档帮助用户设置一个受 OAuth 保护的 AWS 函数，用于连接 GPT Action 以及一个示例应用。本示例使用 AWS SAM（Serverless Application Model）来搭建 AWS 栈。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的能力连接到 AWS 函数。这使你能够连接 AWS 中的任意服务，并在其上运行代码/应用。具体可以体现在以下几个方面：

- 访问第三方服务，例如 AWS Redshift、AWS DynamoDB、AWS S3 等等！
- 支持对来自 API 的文本响应进行预处理（例如突破上下文限制、添加上下文或元数据）。
- 支持返回文件，而不是从第三方 API 获取文本。这可以用于提供 CSV 文件进行数据分析，或者返回 PDF 文件，ChatGPT 会像对待上传文件一样处理它。 


**用例示例**： 
- 用户需要在 Redshift 中查询数据，但需要在 ChatGPT 和 Redshift 之间有一个中间件应用来返回文件（以保证数据分析的数据准确性，并处理大量数据）
- 用户已经在 AWS 函数中构建了若干步骤，需要能够通过 ChatGPT 启动该流程。

## 应用信息与前置条件

我们将利用 AWS Lambda 服务创建一个中间件函数。你可以访问以下链接来熟悉这套技术栈： 

- Lambda 网站：https://aws.amazon.com/lambda/
- Lambda 文档：https://docs.aws.amazon.com/lambda/
- AWS SAM 文档：https://docs.aws.amazon.com/serverless-application-model/

### 前置条件

开始之前，请确保你有一个 AWS 控制台账户，并具备创建以下资源的权限：Lambda Function、S3 Buckets、Application Stack、Cognito User Pool、Cognito User Pool App Clients、API Gateway、Lambda roles、CloudFormation stacks（看起来很多，但这些服务的创建都是自动化的！）。

## 创建 AWS Lambda 函数

要创建 AWS 函数，你可以使用 AWS SAM。SAM 模板的示例可以在[这里](https://github.com/pap-openai/redshift-middleware/blob/main/template.yaml)找到 [0]。

该模板包括：
- 一个 User Pool 和 User Pool Client，用于 OAuth
- 一个 Cognito Authorizer，确保该函数只能被经过身份验证的用户调用
- 将 Lambda 函数映射到现有的 VPC（便于连接其他 AWS 服务）
- 包含可动态设置的参数（例如凭据/变量）
- 一个将 HTTP 路由映射到函数的 API Gateway

这段代码仅供参考，帮助你快速上手，不需要预先存在的 AWS 资源。如果你已有用户池，我们建议映射现有用户池而不是新建；同时，如果你需要利用其他 AWS 资源，建议将 Lambda 部署在能够访问这些资源的 VPC 中。你可以在 [RedShift cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_aws_function) 中看到此类配置的示例。

Cognito Authorizer 是确保你的函数只能被经过身份验证的用户调用/访问的关键，因此请务必根据你的环境正确配置它。

[0]
```
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: >
  aws-middleware

  AWS middleware function

Parameters:
  CognitoUserPoolName:
    Type: String
    Default: MyCognitoUserPool
  CognitoUserPoolClientName:
    Type: String
    Default: MyCognitoUserPoolClient

Resources:
  MyCognitoUserPool:
    Type: AWS::Cognito::UserPool
    Properties:
      UserPoolName: !Ref CognitoUserPoolName
      Policies:
        PasswordPolicy:
          MinimumLength: 8
      UsernameAttributes:
        - email
      Schema:
        - AttributeDataType: String
          Name: email
          Required: false

  MyCognitoUserPoolClient:
    Type: AWS::Cognito::UserPoolClient
    Properties:
      UserPoolId: !Ref MyCognitoUserPool
      ClientName: !Ref CognitoUserPoolClientName
      GenerateSecret: true

  MiddlewareApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      Cors: "'*'"
      Auth:
        DefaultAuthorizer: MyCognitoAuthorizer
        Authorizers:
          MyCognitoAuthorizer:
            AuthorizationScopes:
              - openid
              - email
              - profile
            UserPoolArn: !GetAtt MyCognitoUserPool.Arn
        
  MiddlewareFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: aws-middleware/
      Handler: app.lambda_handler
      Runtime: python3.11
      Timeout: 45
      Architectures:
        - x86_64
      Events:
        SqlStatement:
          Type: Api
          Properties:
            Path: /my_route
            Method: post
            RestApiId: !Ref MiddlewareApi

Outputs:
  MiddlewareApi:
    Description: "API Gateway endpoint URL for Prod stage for SQL Statement function"
    Value: !Sub "https://${MiddlewareApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/my_route"
  MiddlewareFunction:
    Description: "SQL Statement Lambda Function ARN"
    Value: !GetAtt MiddlewareFunction.Arn
  MiddlewareFunctionIamRole:
    Description: "Implicit IAM Role created for SQL Statement function"
    Value: !GetAtt MiddlewareFunctionRole.Arn
  CognitoUserPoolArn:
    Description: "ARN of the Cognito User Pool"
    Value: !GetAtt MyCognitoUserPool.Arn
```

你可以克隆 openai-cookbook 仓库，并从 `lambda-middleware` 目录中获取示例 Python 代码和 SAM 模板：

```
git clone https://github.com/pap-openai/aws-lambda-middleware
cd lambda-middleware
```

要构建并部署你的函数，请在该目录下运行以下命令

```
sam build
sam deploy --template-file template.yaml --stack-name aws-middleware --capabilities CAPABILITY_IAM
```

部署完成后，你可以在 AWS Lambda 上查看该应用：

![../../../images/aws_lambda_1.png](../../../images/aws_lambda_1.png)

你可以在不进行任何身份验证的情况下运行一条 curl 命令，以确认未经身份验证时无法访问该函数：

```
curl -d {} <middleware_api_output_url_from_deploy_command>
```

它应该返回 `{"message":"Unauthorized"}`。

## 在 AWS Cognito 中设置身份验证

_可选：仅当你创建了新用户池而非使用现有用户池时，才需要执行这些步骤_ 

让我们在新建的用户池中创建一个用户。为此，从部署命令的输出中获取 CognitoUserPoolArn，并取 "/" 之后的值，其格式应为：`your-region_xxxxx`。

```
aws cognito-idp admin-create-user \
    --user-pool-id "your-region_xxxxx" \
    --username johndoe@example.com \
    --user-attributes Name=email,Value=johndoe@example.com \
    --temporary-password "TempPassword123"
```

现在让我们确保创建一个可用于登录的网页/域名。前往 AWS Cognito，选择新创建的用户池，并进入 App Integration 选项卡：

![../../../images/aws_lambda_3.png](../../../images/aws_lambda_3.png)

点击 "Domains"，然后点击 "Create Cognito Domain" 来创建 Cognito 域名

![../../../images/aws_lambda_8.png](../../../images/aws_lambda_8.png)

在用户池的 App Integration 页面上向下滚动到 `App client list`：

![../../../images/aws_lambda_9.png](../../../images/aws_lambda_9.png)

选择你的应用客户端并编辑 Hosted UI：

![../../../images/aws_lambda_10.png](../../../images/aws_lambda_10.png)

然后添加回调 URL、授权方案（Authorization Scheme）和 OAuth 权限范围：

![../../../images/aws_lambda_11.png](../../../images/aws_lambda_11.png)

_请注意，当 ChatGPT 为你的 Action 身份验证生成回调 URL 后，你还需要回到这一步。Postman 的 URL 仅应用于开发目的。_

你可以在 Postman 中测试此连接：在 `<api_url>` 的 Authorization 下，从 AWS 复制粘贴 client_id、client_secret 以及你为认证域名设置的 URL，并确保在 scope 中添加 `openid` 以获取有效的 access_token：

![../../../images/aws_lambda_12.png](../../../images/aws_lambda_12.png)

![../../../images/aws_lambda_13.png](../../../images/aws_lambda_13.png)

如果你现在使用刚获取的 access_token 在 Postman 上发起请求，就会得到一个表示成功的 JSON 返回：

![../../../images/aws_lambda_14.png](../../../images/aws_lambda_14.png)

## 在 ChatGPT 中创建 Action

现在让我们将其集成到 ChatGPT 中。

创建一个 Action，并复制粘贴以下规范：

```
openapi: 3.1.0
info:
  title: Success API
  description: API that returns a success message.
  version: 1.0.0
servers:
  - url: https://3ho5n15aef.execute-api.us-east-1.amazonaws.com/Prod
    description: Main production server
paths:
  /my_route:
    post:
      operationId: postSuccess
      summary: Returns a success message.
      description: Endpoint to check the success status.
      responses:
        '200':
          description: A JSON object indicating success.
          content:
            application/json:
              schema:
                type: object
                properties:
                  success:
                    type: boolean
                    example: true
```

如果你尝试测试该 Action（可以点击 "Test" 按钮），会看到返回 401，因为你尚未通过身份验证。

现在让我们为该 Action 添加身份验证。

点击 Authentication > OAuth。
接下来我们需要获取 AWS Cognito 的变量。进入你的 User Pool > User Pool App Client，在那里可以获取 client ID 和 client Secret。

![../../../images/aws_lambda_15.png](../../../images/aws_lambda_15.png)

将这些值复制粘贴到 ChatGPT 中。现在让我们添加 Token URL。

在你的 User Pool 中，可以找到之前为托管域名创建的 URL。

![../../../images/aws_lambda_16.png](../../../images/aws_lambda_16.png)

我们将使用该 URL，并在其后追加 [AWS 的 OAuth 路由](https://docs.aws.amazon.com/cognito/latest/developerguide/federation-endpoints.html)。

- token：`<your_url>/oauth2/token`
- authorization：`<your_url>/oauth2/authorize`

将它们复制粘贴到 ChatGPT 中。

在 scope 中添加 openid，然后点击 Save。

## 使用 ChatGPT URL 配置 Cognito

现在回到你的 GPT（退出 Action 子视图），你会看到 ChatGPT 为身份验证提供的回调 URL：

![../../../images/aws_lambda_17.png](../../../images/aws_lambda_17.png)

获取该 URL，编辑 User Pool App client 的 Hosted UI 并保存更改：

![../../../images/aws_lambda_18.png](../../../images/aws_lambda_18.png)

## 测试函数

现在你可以再次测试该 Action：

![../../../images/aws_lambda_19.png](../../../images/aws_lambda_19.png)

你将被重定向到 AWS Cognito 页面，可以使用之前设置的凭据登录。

如果你现在让 GPT 运行同一个 Action，它会正确作答，因为你现在已通过身份验证，能够运行该函数了！

![../../../images/aws_lambda_20.png](../../../images/aws_lambda_20.png)

# 结论

你现在已经在 ChatGPT 中设置了一个能够以经过身份验证的方式与你在 AWS 中的应用通信的 Action！本 cookbook 演示了如何使用用户名/密码从零创建 Cognito 用户池，不过我们建议根据你的需求来配置 Cognito（例如将你自己的 IDP 接入 Cognito）。

此外，该函数没有连接任何其他服务，而这正是能够以安全方式与 AWS Lambda 函数通信的优势所在。因此，你可以调整代码和 AWS SAM 模板以满足自己的需求。一个更复杂的函数示例是 Redshift，它按照这些步骤创建函数和身份验证，但代码/部署方式有所不同。